# Relive — Phase 0: reconstruct a place from OLD PHOTOS

Companion to `phase0_instantsplat_colab.ipynb` (which takes video). Same engine, tuned for a handful of **existing, uncontrolled photographs** — the "revisit a place you can't go back to" case.

**What's different from the video notebook, and why:**

| | video notebook | this one |
|---|---|---|
| frame culling | aggressive (sharpest 12 of ~900) | **keep everything** — you may only have 6 photos |
| EXIF rotation | n/a | **applied** (OpenCV ignores it; a sideways photo wrecks reconstruction) |
| `--focal_avg` | on (one camera, one focal) | **off by default** (different cameras = different focals) |
| iterations | 1000 | 3000 (fewer views benefit from longer training) |

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

**Expect this to be harder than video.** Old photos have no guaranteed overlap, inconsistent lighting, different cameras and eras. That difficulty is the point: if it works at all it's a moat, because every competitor assumes fresh capture.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Torch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch to a GPU runtime')

## 2. Clone InstantSplat + download the MASt3R checkpoint

In [ ]:
%cd /content
![ -d InstantSplat ] || git clone --recursive https://github.com/NVlabs/InstantSplat.git
%cd /content/InstantSplat
!mkdir -p mast3r/checkpoints/
![ -f mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth ] || wget -q --show-progress https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth -P mast3r/checkpoints/
import os
print('checkpoint present:', os.path.exists('mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'))

## 3. Install dependencies (compiles the CUDA splat kernels, a few minutes)

`curope` is skipped on purpose — Colab's PyTorch rejects its old dispatch API, and CroCo/MASt3R falls back to a pure-PyTorch RoPE automatically. You'll see a "using a slow pytorch version" warning later; that is expected, not an error.

In [ ]:
%cd /content/InstantSplat
!pip install -q -r requirements.txt
!pip install -q submodules/simple-knn
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/fused-ssim
print('install complete (curope intentionally skipped)')

## 4. Upload your photos

Run, then select **all the photos of one place**. Guidance:

- **One place per run.** Don't mix the kitchen and the garden.
- **Overlap is everything.** Photos from different spots in the same room work; photos of unrelated corners sharing no common surfaces will fail.
- **6–20 photos** is the sweet spot. Below ~5 is very hard; above 20 may exhaust the T4.
- Scans of printed photos are fine — crop off white borders first.

In [ ]:
import os, glob
from google.colab import files

INPUT_DIR = '/content/input'
os.makedirs(INPUT_DIR, exist_ok=True)
for f in glob.glob(INPUT_DIR + '/*'):
    os.remove(f)
%cd /content/input
print('Select all photos of ONE place:')
uploaded = files.upload()
%cd /content/InstantSplat
print('Received', len(uploaded), 'file(s)')

## 5. Prepare the photos

Applies EXIF rotation (critical — OpenCV silently ignores it, and one sideways photo can wreck the solve), downsizes oversized images, and reports each one so you can spot problems. Keeps **all** photos unless you exceed the cap.

In [ ]:
import os, glob, numpy as np, cv2
from PIL import Image, ImageOps

# ---- tunables ----
MAX_WIDTH  = 1024   # downsize anything wider; never upscales small old photos
MAX_PHOTOS = 20     # MASt3R pairs grow ~quadratically; cap to fit the T4
# ------------------

SCENE = 'relive_photos'
SCENE_DIR = '/content/InstantSplat/assets/' + SCENE + '/images'
os.makedirs(SCENE_DIR, exist_ok=True)
for f in glob.glob(SCENE_DIR + '/*'):
    os.remove(f)

exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')
paths = [p for p in sorted(glob.glob('/content/input/*')) if p.lower().endswith(exts)]
if not paths:
    raise RuntimeError('No photos found in /content/input - re-run cell 4.')

print(str(len(paths)) + ' photo(s) found')
print('')
print('file'.ljust(26), 'original'.rjust(12), 'used'.rjust(12), 'exif'.rjust(8), 'sharp'.rjust(8))
print('-' * 72)

prepared = []
for p in paths:
    im0 = Image.open(p)
    orig = im0.size
    im = ImageOps.exif_transpose(im0).convert('RGB')   # honour EXIF rotation
    exif_note = 'rotated' if im.size != orig else '-'
    if im.width > MAX_WIDTH:
        im = im.resize((MAX_WIDTH, round(im.height * MAX_WIDTH / im.width)), Image.LANCZOS)
    g = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY)
    sharp = cv2.Laplacian(g, cv2.CV_64F).var()
    prepared.append((sharp, os.path.basename(p), im))
    print(os.path.basename(p)[:26].ljust(26), str(orig).rjust(12), str(im.size).rjust(12), exif_note.rjust(8), ('%.1f' % sharp).rjust(8))

if len(prepared) > MAX_PHOTOS:
    print('')
    print(str(len(prepared)) + ' photos exceeds cap of ' + str(MAX_PHOTOS) + '; keeping the sharpest ' + str(MAX_PHOTOS))
    prepared.sort(key=lambda t: t[0], reverse=True)
    prepared = prepared[:MAX_PHOTOS]

for i in range(len(prepared)):
    prepared[i][2].save(SCENE_DIR + '/%04d.jpg' % i, quality=95)

N_VIEWS = len(prepared)
sizes = sorted(set(im.size for _, _, im in prepared))
print('')
print('wrote ' + str(N_VIEWS) + ' photo(s) to ' + SCENE_DIR)
print('resolutions in use:', sizes)
print('')
if len(sizes) > 1:
    print('>> MIXED RESOLUTIONS - almost certainly different cameras.')
    print('>> Keep SAME_CAMERA = False in the next cell (do NOT average focal lengths).')
else:
    print('>> Uniform resolution. If these all came from ONE camera at one zoom,')
    print('>> you may set SAME_CAMERA = True in the next cell for a better pose solve.')

## 6. Run the reconstruction

`SAME_CAMERA` is the knob that matters here. `--focal_avg` forces one shared focal length across all images — correct for video from a single phone, **wrong** for a pile of photos from different cameras. Leave it `False` unless you're sure.

In [ ]:
import os, glob, shutil
%cd /content/InstantSplat

# ---- the knobs that matter ----
SAME_CAMERA   = False   # True ONLY if every photo is from one camera at one zoom
GS_TRAIN_ITER = 3000    # more than the video notebook; few views benefit from longer training
# -------------------------------

# PyTorch >=2.6 refuses the MASt3R checkpoint under weights_only=True. Restore old behaviour.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
OLD = "torch.load(model_path, map_location='cpu')"
NEW = "torch.load(model_path, map_location='cpu', weights_only=False)"
for path in ['mast3r/model.py', 'dust3r/model.py']:
    p = '/content/InstantSplat/' + path
    if os.path.exists(p):
        s = open(p).read()
        if OLD in s:
            open(p, 'w').write(s.replace(OLD, NEW))
            print('patched torch.load in', path)

SCENE = 'relive_photos'
SOURCE_PATH = '/content/InstantSplat/assets/' + SCENE + '/'   # scene dir, NOT images/
IMAGE_DIR   = SOURCE_PATH + 'images'
for d in glob.glob(SOURCE_PATH + 'sparse_*') + glob.glob(IMAGE_DIR + '/sparse_*'):
    shutil.rmtree(d, ignore_errors=True)

N_VIEWS = len([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
MODEL_PATH = '/content/InstantSplat/output_infer/' + SCENE + '/' + str(N_VIEWS) + '_views'
os.makedirs(MODEL_PATH, exist_ok=True)
FOCAL = ' --focal_avg' if SAME_CAMERA else ''
print('views:', N_VIEWS, '| iters:', GS_TRAIN_ITER, '| focal_avg:', SAME_CAMERA)

# Stage 1 - MASt3R poses + sparse cloud. No --infer_video: its pose interpolation runs
# BEFORE the sparse-file save and can get killed, leaving stage 2 with nothing to read.
!python -W ignore ./init_geo.py -s "$SOURCE_PATH" -m "$MODEL_PATH" --n_views $N_VIEWS --co_vis_dsp --conf_aware_ranking $FOCAL

# Stage 2 - train the gaussians. THIS is what produces point_cloud.ply.
!python ./train.py -s "$SOURCE_PATH" -m "$MODEL_PATH" -r 1 --n_views $N_VIEWS --iterations $GS_TRAIN_ITER --pp_optimizer --optim_pose

# Stage 3 - optional render of training views; the .ply already exists after stage 2.
!python ./render.py -s "$SOURCE_PATH" -m "$MODEL_PATH" -r 1 --n_views $N_VIEWS --iterations $GS_TRAIN_ITER

print('')
print('--- run finished ---')

## 7. Find and download the trained splat

Two very different `.ply` files exist. You want the **trained splat** (binary, has `f_dc_0`/`opacity`/`scale_0`), not the init cloud (`points3D.ply` — x/y/z only, no colour, renders as blue-olive dots and is *not* a result).

In [ ]:
import glob
from google.colab import files

print('type'.ljust(15), 'vertices'.rjust(12), ' path')
print('-' * 80)
best = None
for p in sorted(glob.glob('/content/InstantSplat/**/*.ply', recursive=True)):
    head = open(p, 'rb').read(2500).decode('ascii', 'ignore')
    nrow = [l for l in head.split('\n') if l.startswith('element vertex')]
    n = int(nrow[0].split()[-1]) if nrow else 0
    is_gs = ('f_dc_0' in head) and ('opacity' in head) and ('scale_0' in head)
    print(('TRAINED SPLAT' if is_gs else 'init cloud').ljust(15), ('{:,}'.format(n)).rjust(12), ' ' + p)
    if is_gs and (best is None or n > best[1]):
        best = (p, n)

print('')
if best:
    print('Downloading trained splat:', best[0], '(' + '{:,}'.format(best[1]) + ' gaussians)')
    files.download(best[0])
else:
    print('No trained splat found - stage 2 did not save. Check the cell 6 output above.')

## 8. Judge the result — and what it means

Open the `.ply` at **https://superspl.at/editor** (drag it in). If it renders as dots, that's the centers/debug view mode, not your data.

**Read the outcome as a business experiment, not just a render:**

- **Recognisable place, soft or incomplete** → *the thesis works.* Sparse uncontrolled photos can reconstruct. Gaps then become a generative-fill problem, which is exactly the moat: fabricating plausible structure is acceptable here because the alternative is nothing, and nobody can go back and re-shoot.
- **Geometry roughly right but badly distorted** → poses solved, views too thin. Try `SAME_CAMERA` the other way, more iterations, or add photos with more overlap.
- **Incoherent mush** → the photos share too little overlap for correspondence. That's the honest failure mode, and it means the product needs a minimum-overlap check at upload time rather than more compute.

**Calibration worth doing:** run one easy case (several photos of one room, same day, same camera) and one hard case (a few genuinely old photos, different eras). The gap between those two results is your real product spec.